# 11b — Dipole Scatter Plot: psfFlux vs (scienceFlux − templateFlux)

This notebook produces diagnostic scatter plots to characterise dipole-flagged DIA sources
in the psfFlux / (scienceFlux − templateFlux) plane.

**Physical motivation:**
- `psfFlux` is the flux measured in the **difference image** (science − template PSF-fit).
- `scienceFlux − templateFlux` is the naive pixel-sum difference between the two images.
- For a perfect PSF subtraction these two quantities should be equal.
- Dipole-flagged sources tend to lie off the 1:1 line, revealing PSF-matching failures.

**Plot design (2 × 3 subplots, one figure per stellar category):**
- Row 0 : bands u, g, r
- Row 1 : bands i, z, y
- X axis : `psfFlux` (nJy) with errorbar `psfFluxErr`
- Y axis : `scienceFlux − templateFlux` (nJy) with propagated errorbar
- Marker style : open circle (○) = dipole flagged; filled disk (●) = not a dipole
- Marker colour : AB magnitude of `scienceFlux` (diverging colormap)
- Colorbar shown at the right of each subplot

**Stellar categories:**
- `gaia_star_stable_hq` — Gaia stable HQ (primary calibration targets)
- `gaia_nophotgstar_stable_unknown_parallax` — Gaia stable, no photometric solution
- `gaia_star_variable` — Gaia variable stars (control group)

**Author:** Sylvie Dagoret-Campagne (IJCLab/IN2P3/CNRS, Université Paris-Saclay)  
**Creation Date:** 2026-05-16  
**Notebook tag:** 11b

## 1. Imports & configuration

In [ ]:
import pandas as pd
import numpy as np
import os
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import warnings
from IPython.display import display

warnings.filterwarnings("ignore")

print(f"pandas  : {pd.__version__}")
print(f"numpy   : {np.__version__}")
print(f"matplotlib: {mpl.__version__}")

In [ ]:
try:
    import ipympl  # noqa: F401

    %matplotlib widget
    print("ipympl → interactive backend")
except ImportError:
    %matplotlib inline
    print("ipympl not found → inline backend")

In [ ]:
# ── Paths ─────────────────────────────────────────────────────────────────────
NB_TAG = "FINK_BLOCK_LC_01"
DIR_DATA = f"data_{NB_TAG}"  # input from notebook 01
DIR_FIGS = f"figs_{NB_TAG}_11"  # shared output directory with notebook 11
os.makedirs(DIR_FIGS, exist_ok=True)

# ── Photometric bands in display order (2 rows × 3 columns) ──────────────────
BANDS = list("ugrizy")
BANDS_ROW0 = list("ugr")
BANDS_ROW1 = list("izy")

BAND_COLORS = {
    "u": "#9b59b6",
    "g": "#2ecc71",
    "r": "#e74c3c",
    "i": "#e67e22",
    "z": "#3498db",
    "y": "#795548",
}

# ── Stellar categories ────────────────────────────────────────────────────────
CATEGORIES = {
    "gaia_star_stable_hq": {
        "label": "Gaia stable HQ",
        "color": "steelblue",
    },
    "gaia_nophotgstar_stable_unknown_parallax": {
        "label": "Gaia stable no-phot",
        "color": "seagreen",
    },
    "gaia_star_variable": {
        "label": "Gaia variable",
        "color": "firebrick",
    },
}

# ── AB magnitude calibration ──────────────────────────────────────────────────
AB_FLUX_ZERO = 3631e9  # nJy  (m_AB = -2.5 log10(f_nJy / AB_FLUX_ZERO))

# ── Scatter plot options ──────────────────────────────────────────────────────
# Colormap for marker colour (AB mag of scienceFlux)
CMAP_NAME = "viridis_r"  # bright stars = yellow; faint = purple
# scienceFlux AB mag range for the colormap
MAG_VMIN = 15.0
MAG_VMAX = 27.0
# Marker sizes
MS_NODIP = 20  # filled disk (non-dipole)
MS_DIP = 40  # open circle (dipole)
ALPHA_NODIP = 0.35
ALPHA_DIP = 0.85

# ── Plot style ────────────────────────────────────────────────────────────────
plt.rcParams.update(
    {
        "figure.dpi": 120,
        "axes.grid": True,
        "grid.alpha": 0.25,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "font.size": 9,
    }
)


def savefig(name):
    """Save the current figure as PDF and PNG in DIR_FIGS."""
    for ext in ("pdf", "png"):
        plt.savefig(os.path.join(DIR_FIGS, f"{name}.{ext}"), bbox_inches="tight")
    print(f"  -> saved {name}.{{pdf,png}}")


print(f"Data dir : {os.path.abspath(DIR_DATA)}")
print(f"Figs dir : {os.path.abspath(DIR_FIGS)}")

## 2. Utility functions

In [ ]:
def flux_nJy_to_mag_AB(flux_nJy):
    """Convert flux in nJy to AB magnitude. Returns NaN for non-positive flux."""
    f = np.asarray(flux_nJy, dtype=float)
    with np.errstate(invalid="ignore", divide="ignore"):
        return np.where(f > 0, -2.5 * np.log10(f / AB_FLUX_ZERO), np.nan)


def parse_dipole_bool(series: pd.Series) -> pd.Series:
    """Coerce a dipole-flag column (bool / int / str) to a boolean Series."""

    def _cast(val):
        if isinstance(val, bool):
            return val
        if isinstance(val, (int, float)):
            return bool(val)
        if isinstance(val, str):
            return val.strip().lower() in ("true", "1", "yes")
        return False

    return series.apply(_cast)


print("Utility functions defined.")

## 3. Load parquet data

Same loading logic as notebook 11.  We need `psfFlux`, `psfFluxErr`,
`scienceFlux`, `scienceFluxErr`, `templateFlux`, `templateFluxErr`,
and `isDipole`.  Missing error columns default to zero (plotted as upper limits
of zero extent).

In [ ]:
data = {}  # data[cat_key] = cleaned DataFrame

for cat in CATEGORIES:
    fpath = os.path.join(DIR_DATA, f"{cat}_src.parquet")
    if not os.path.exists(fpath):
        print(f"[SKIP] {cat}: file not found ({fpath})")
        continue

    df = pd.read_parquet(fpath)

    # ── Mandatory columns ─────────────────────────────────────────────────
    required = ["r:midpointMjdTai", "r:psfFlux", "r:psfFluxErr", "r:band"]
    missing = [c for c in required if c not in df.columns]
    if missing:
        print(f"[WARN] {cat}: missing columns {missing} — skipped")
        continue

    df = df.dropna(subset=["r:psfFlux", "r:psfFluxErr"]).reset_index(drop=True)

    # ── Dipole flag ────────────────────────────────────────────────────────
    if "r:isDipole" in df.columns:
        df["is_dipole"] = parse_dipole_bool(df["r:isDipole"].fillna(False))
    else:
        print(f"[WARN] {cat}: no r:isDipole column — all set to False")
        df["is_dipole"] = False

    # ── scienceFlux ────────────────────────────────────────────────────────
    if "r:scienceFlux" in df.columns:
        df["scienceFlux"] = df["r:scienceFlux"].astype(float)
        df["scienceFluxErr"] = (
            df["r:scienceFluxErr"].astype(float) if "r:scienceFluxErr" in df.columns else 0.0
        )
    else:
        print(f"[INFO] {cat}: r:scienceFlux absent — using |psfFlux| as proxy")
        df["scienceFlux"] = np.abs(df["r:psfFlux"].values)
        df["scienceFluxErr"] = df["r:psfFluxErr"].values

    # ── templateFlux ───────────────────────────────────────────────────────
    if "r:templateFlux" in df.columns:
        df["templateFlux"] = df["r:templateFlux"].astype(float)
        df["templateFluxErr"] = (
            df["r:templateFluxErr"].astype(float) if "r:templateFluxErr" in df.columns else 0.0
        )
    else:
        print(f"[INFO] {cat}: r:templateFlux absent — template terms set to 0")
        df["templateFlux"] = 0.0
        df["templateFluxErr"] = 0.0

    # ── Derived quantities ─────────────────────────────────────────────────
    df["diff_flux"] = df["scienceFlux"] - df["templateFlux"]  # Y axis
    df["diff_fluxErr"] = np.sqrt(df["scienceFluxErr"] ** 2 + df["templateFluxErr"] ** 2)  # Y error

    df["psfFlux"] = df["r:psfFlux"].astype(float)  # X axis
    df["psfFluxErr"] = df["r:psfFluxErr"].astype(float)  # X error

    # AB magnitude of scienceFlux — used for marker colour
    df["mag_science"] = flux_nJy_to_mag_AB(df["scienceFlux"].values)

    data[cat] = df
    lbl = CATEGORIES[cat]["label"]
    n_tot = len(df)
    n_dip = int(df["is_dipole"].sum())
    print(f"  {lbl:40s}  {n_tot:6d} rows  {n_dip:5d} dipoles  ({100 * n_dip / n_tot:.1f}%)")

CATS_OK = list(data.keys())
print(f"\nCategories loaded: {CATS_OK}")

## 4. Quick inspection

In [ ]:
if CATS_OK:
    cat0 = CATS_OK[0]
    cols_interest = [
        "r:band",
        "psfFlux",
        "psfFluxErr",
        "scienceFlux",
        "scienceFluxErr",
        "templateFlux",
        "templateFluxErr",
        "diff_flux",
        "diff_fluxErr",
        "mag_science",
        "is_dipole",
    ]
    cols_present = [c for c in cols_interest if c in data[cat0].columns]
    print(f"Preview — {CATEGORIES[cat0]['label']}:")
    display(data[cat0][cols_present].head(8))

## 5. Scatter-plot function

### Design choices

| Element | Dipole (`isDipole=True`) | Non-dipole |
|---------|--------------------------|------------|
| Marker  | open circle `'o'`, `facecolors='none'` | filled disk `'o'` |
| Size    | larger (`MS_DIP`) | smaller (`MS_NODIP`) |
| Alpha   | `ALPHA_DIP` (0.85) | `ALPHA_NODIP` (0.35) |
| Z-order | top | bottom |

Colour is driven by `mag_science` (AB mag of `scienceFlux`):
bright → yellow, faint → purple (viridis_r). A dedicated colorbar is drawn
at the right of **each** subplot via `plt.colorbar(sm, ax=ax)` so the scale
is always visible regardless of page layout.

In [ ]:
def plot_dipole_scatter_2x3(
    df,
    cat_label,
    cat_color,
    cmap_name=CMAP_NAME,
    mag_vmin=MAG_VMIN,
    mag_vmax=MAG_VMAX,
    flux_lim=None,
    savename=None,
):
    """
    Produce a 2 × 3 scatter-plot grid for one stellar category.

    X axis : psfFlux                      (nJy, with psfFluxErr errorbars)
    Y axis : scienceFlux − templateFlux   (nJy, with propagated errorbars)
    Colour : AB magnitude of scienceFlux  (per-subplot colormap + colorbar)
    Style  : open circle  = dipole flagged
             filled disk  = not a dipole

    Parameters
    ----------
    df          : DataFrame  — loaded src data for this category
    cat_label   : str        — category human label for suptitle
    cat_color   : str        — category colour (suptitle colour)
    cmap_name   : str        — matplotlib colormap name
    mag_vmin    : float      — colormap minimum AB magnitude (bright end)
    mag_vmax    : float      — colormap maximum AB magnitude (faint end)
    flux_lim    : float|None — symmetric axis limit in nJy; auto if None
    savename    : str|None   — stem for PDF+PNG output files
    """
    cmap = cm.get_cmap(cmap_name)
    norm = mpl.colors.Normalize(vmin=mag_vmin, vmax=mag_vmax)
    sm = cm.ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([])  # required for standalone colorbar

    fig, axes = plt.subplots(
        2,
        3,
        figsize=(5.5 * 3, 5.0 * 2),
        squeeze=False,
    )
    fig.suptitle(
        f"{cat_label}\n"
        r"psfFlux  vs  scienceFlux $-$ templateFlux",
        fontsize=12,
        fontweight="bold",
        y=1.01,
        color=cat_color,
    )

    for row_idx, bands_row in enumerate([BANDS_ROW0, BANDS_ROW1]):
        for col_idx, band in enumerate(bands_row):
            ax = axes[row_idx][col_idx]
            bcolor = BAND_COLORS[band]

            df_b = df[df["r:band"] == band].copy()
            df_b = df_b.dropna(subset=["psfFlux", "diff_flux", "mag_science"])

            df_nodip = df_b[~df_b["is_dipole"]]
            df_dip = df_b[df_b["is_dipole"]]
            n_nodip = len(df_nodip)
            n_dip = len(df_dip)

            def mag_to_rgba(mag_series):
                """Map AB magnitudes Series to RGBA arrays via the colormap."""
                clipped = np.clip(mag_series.values, mag_vmin, mag_vmax)
                return cmap(norm(clipped))

            # ── Non-dipole: filled disks, small, transparent ─────────────
            if n_nodip > 0:
                rgba_nd = mag_to_rgba(df_nodip["mag_science"])
                ax.errorbar(
                    df_nodip["psfFlux"].values,
                    df_nodip["diff_flux"].values,
                    xerr=df_nodip["psfFluxErr"].values,
                    yerr=df_nodip["diff_fluxErr"].values,
                    fmt="none",
                    ecolor="silver",
                    elinewidth=0.4,
                    alpha=0.4,
                    zorder=1,
                )
                ax.scatter(
                    df_nodip["psfFlux"].values,
                    df_nodip["diff_flux"].values,
                    s=MS_NODIP,
                    c=rgba_nd,
                    marker="o",
                    alpha=ALPHA_NODIP,
                    linewidths=0,
                    zorder=2,
                    label=f"no dipole (N={n_nodip})",
                )

            # ── Dipole: open circles, larger, opaque edge ─────────────────
            if n_dip > 0:
                rgba_dip = mag_to_rgba(df_dip["mag_science"])
                ax.errorbar(
                    df_dip["psfFlux"].values,
                    df_dip["diff_flux"].values,
                    xerr=df_dip["psfFluxErr"].values,
                    yerr=df_dip["diff_fluxErr"].values,
                    fmt="none",
                    ecolor="tomato",
                    elinewidth=0.7,
                    alpha=0.7,
                    zorder=3,
                )
                ax.scatter(
                    df_dip["psfFlux"].values,
                    df_dip["diff_flux"].values,
                    s=MS_DIP,
                    c="none",
                    edgecolors=rgba_dip,
                    marker="o",
                    linewidths=1.5,
                    alpha=ALPHA_DIP,
                    zorder=4,
                    label=f"dipole (N={n_dip})",
                )

            # ── Symmetric axis limits ─────────────────────────────────────
            if flux_lim is not None:
                ax.set_xlim(-flux_lim, flux_lim)
                ax.set_ylim(-flux_lim, flux_lim)
            else:
                all_x = df_b["psfFlux"].values
                all_y = df_b["diff_flux"].values
                if len(all_x) > 0:
                    max_range = np.nanmax(np.abs(np.concatenate([all_x, all_y])))
                    if np.isfinite(max_range) and max_range > 0:
                        lim = max_range * 1.10
                        ax.set_xlim(-lim, lim)
                        ax.set_ylim(-lim, lim)

            # ── Reference line y = x ─────────────────────────────────────
            xlim = ax.get_xlim()
            x_ref = np.linspace(*xlim, 100)
            ax.plot(x_ref, x_ref, "k--", lw=0.8, alpha=0.5, label="y = x", zorder=0)
            ax.axhline(0, color="grey", lw=0.6, ls=":", alpha=0.5)
            ax.axvline(0, color="grey", lw=0.6, ls=":", alpha=0.5)

            # ── Colorbar (one per subplot) ────────────────────────────────
            cb = plt.colorbar(sm, ax=ax, fraction=0.046, pad=0.04)
            cb.set_label("scienceFlux (AB mag)", fontsize=7)
            cb.ax.tick_params(labelsize=6)
            cb.ax.invert_yaxis()  # low AB mag (bright) at bottom

            # ── Labels & cosmetics ────────────────────────────────────────
            ax.set_title(
                f"band  {band}   [{n_nodip + n_dip} sources]",
                color=bcolor,
                fontweight="bold",
                fontsize=10,
            )
            ax.set_xlabel(r"psfFlux  (nJy)", fontsize=9)
            ax.set_ylabel(r"scienceFlux $-$ templateFlux  (nJy)", fontsize=9)
            leg = ax.legend(fontsize=7, loc="upper left", framealpha=0.7)
            for lh in leg.legend_handles:
                lh.set_alpha(0.9)

    plt.tight_layout()
    if savename:
        savefig(savename)
    plt.show()


print("plot_dipole_scatter_2x3() defined.")

## 6. Run scatter plots — one figure per stellar category

The three figures below show all DIA sources in the
**psfFlux vs (scienceFlux − templateFlux)** plane.

**What to look for:**
- Points near the identity line (y = x) indicate consistent flux measurements between
  the PSF-fit and the naive pixel-difference.
- Dipole-flagged points (open circles) departing from y = x reveal PSF-matching
  artefacts that inflate the psfFlux relative to the true brightness change.
- A bright-magnitude (yellow) cluster of open circles near x ≈ 0, y ≈ 0 would
  suggest that bright stars are systematically creating near-zero but flagged residuals.

In [ ]:
# Optional: set a common symmetric flux range in nJy (None → per-category auto range)
# Uncomment and adjust if you want all figures on the same scale for comparison:
# FLUX_LIM = 5000.0   # nJy
FLUX_LIM = None

for cat in CATS_OK:
    lbl = CATEGORIES[cat]["label"]
    color = CATEGORIES[cat]["color"]
    safe = cat.replace("/", "_")
    print(f"\n{'=' * 65}")
    print(f"  {lbl}")
    print(f"{'=' * 65}")
    plot_dipole_scatter_2x3(
        data[cat],
        cat_label=lbl,
        cat_color=color,
        flux_lim=FLUX_LIM,
        savename=f"11b_dipole_scatter_{safe}",
    )

## 7. Zoom-in: flux range ±5σ per band

The full flux range can be dominated by a few bright outliers.
This section repeats the plots with a **robust symmetric range**:
`±5 × (84th–16th percentile half-range)` of `psfFlux` and `diff_flux`, computed per band.
This zoom reveals the bulk structure of the scatter more clearly.

In [ ]:
def plot_dipole_scatter_2x3_zoom(
    df,
    cat_label,
    cat_color,
    n_sigma=5.0,
    cmap_name=CMAP_NAME,
    mag_vmin=MAG_VMIN,
    mag_vmax=MAG_VMAX,
    savename=None,
):
    """
    Same as plot_dipole_scatter_2x3 but with per-band robust axis limits:
    ± n_sigma × robust_std  where robust_std = (p84 - p16) / 2.
    """
    cmap = cm.get_cmap(cmap_name)
    norm = mpl.colors.Normalize(vmin=mag_vmin, vmax=mag_vmax)
    sm = cm.ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([])

    fig, axes = plt.subplots(
        2,
        3,
        figsize=(5.5 * 3, 5.0 * 2),
        squeeze=False,
    )
    fig.suptitle(
        f"{cat_label}  [zoom ±{n_sigma}σ]\n"
        r"psfFlux  vs  scienceFlux $-$ templateFlux",
        fontsize=12,
        fontweight="bold",
        y=1.01,
        color=cat_color,
    )

    for row_idx, bands_row in enumerate([BANDS_ROW0, BANDS_ROW1]):
        for col_idx, band in enumerate(bands_row):
            ax = axes[row_idx][col_idx]
            bcolor = BAND_COLORS[band]

            df_b = df[df["r:band"] == band].copy()
            df_b = df_b.dropna(subset=["psfFlux", "diff_flux", "mag_science"])

            df_nodip = df_b[~df_b["is_dipole"]]
            df_dip = df_b[df_b["is_dipole"]]
            n_nodip = len(df_nodip)
            n_dip = len(df_dip)

            def mag_to_rgba(mag_series):
                clipped = np.clip(mag_series.values, mag_vmin, mag_vmax)
                return cmap(norm(clipped))

            # Robust symmetric axis limit from the joint psfFlux + diff_flux distribution
            all_vals = df_b[["psfFlux", "diff_flux"]].values.ravel()
            all_vals = all_vals[np.isfinite(all_vals)]
            if len(all_vals) > 4:
                p16, p84 = np.percentile(all_vals, [16, 84])
                robust_std = (p84 - p16) / 2.0
                flux_lim = n_sigma * robust_std
            else:
                flux_lim = None

            # Non-dipole
            if n_nodip > 0:
                rgba_nd = mag_to_rgba(df_nodip["mag_science"])
                ax.errorbar(
                    df_nodip["psfFlux"].values,
                    df_nodip["diff_flux"].values,
                    xerr=df_nodip["psfFluxErr"].values,
                    yerr=df_nodip["diff_fluxErr"].values,
                    fmt="none",
                    ecolor="silver",
                    elinewidth=0.4,
                    alpha=0.4,
                    zorder=1,
                )
                ax.scatter(
                    df_nodip["psfFlux"].values,
                    df_nodip["diff_flux"].values,
                    s=MS_NODIP,
                    c=rgba_nd,
                    marker="o",
                    alpha=ALPHA_NODIP,
                    linewidths=0,
                    zorder=2,
                    label=f"no dipole (N={n_nodip})",
                )

            # Dipole
            if n_dip > 0:
                rgba_dip = mag_to_rgba(df_dip["mag_science"])
                ax.errorbar(
                    df_dip["psfFlux"].values,
                    df_dip["diff_flux"].values,
                    xerr=df_dip["psfFluxErr"].values,
                    yerr=df_dip["diff_fluxErr"].values,
                    fmt="none",
                    ecolor="tomato",
                    elinewidth=0.7,
                    alpha=0.7,
                    zorder=3,
                )
                ax.scatter(
                    df_dip["psfFlux"].values,
                    df_dip["diff_flux"].values,
                    s=MS_DIP,
                    c="none",
                    edgecolors=rgba_dip,
                    marker="o",
                    linewidths=1.5,
                    alpha=ALPHA_DIP,
                    zorder=4,
                    label=f"dipole (N={n_dip})",
                )

            # Axis limits and reference lines
            if flux_lim is not None:
                ax.set_xlim(-flux_lim, flux_lim)
                ax.set_ylim(-flux_lim, flux_lim)
                x_ref = np.linspace(-flux_lim, flux_lim, 100)
            else:
                x_ref = np.linspace(*ax.get_xlim(), 100)
            ax.plot(x_ref, x_ref, "k--", lw=0.8, alpha=0.5, label="y = x", zorder=0)
            ax.axhline(0, color="grey", lw=0.6, ls=":", alpha=0.5)
            ax.axvline(0, color="grey", lw=0.6, ls=":", alpha=0.5)

            # Colorbar
            cb = plt.colorbar(sm, ax=ax, fraction=0.046, pad=0.04)
            cb.set_label("scienceFlux (AB mag)", fontsize=7)
            cb.ax.tick_params(labelsize=6)
            cb.ax.invert_yaxis()

            ax.set_title(
                f"band  {band}   [{n_nodip + n_dip} sources]",
                color=bcolor,
                fontweight="bold",
                fontsize=10,
            )
            ax.set_xlabel(r"psfFlux  (nJy)", fontsize=9)
            ax.set_ylabel(r"scienceFlux $-$ templateFlux  (nJy)", fontsize=9)
            leg = ax.legend(fontsize=7, loc="upper left", framealpha=0.7)
            for lh in leg.legend_handles:
                lh.set_alpha(0.9)

    plt.tight_layout()
    if savename:
        savefig(savename)
    plt.show()


print("plot_dipole_scatter_2x3_zoom() defined.")

In [ ]:
for cat in CATS_OK:
    lbl = CATEGORIES[cat]["label"]
    color = CATEGORIES[cat]["color"]
    safe = cat.replace("/", "_")
    print(f"\n{'=' * 65}")
    print(f"  {lbl}  [zoom ±5σ]")
    print(f"{'=' * 65}")
    plot_dipole_scatter_2x3_zoom(
        data[cat],
        cat_label=lbl,
        cat_color=color,
        n_sigma=5.0,
        savename=f"11b_dipole_scatter_zoom_{safe}",
    )

## 8. Marginal statistics: dipole vs non-dipole in the flux-difference plane

For each category and band we compute:
- Median and MAD (median absolute deviation) of `psfFlux` and `diff_flux`
  separately for dipole and non-dipole subsets.
- The ratio `psfFlux / diff_flux` (should be 1 for perfect subtraction).

This table summarises the systematic offset between the two flux estimators
and whether dipole-flagging correlates with larger offsets.

In [ ]:
def mad(x):
    """Median absolute deviation."""
    m = np.nanmedian(x)
    return np.nanmedian(np.abs(x - m))


for cat in CATS_OK:
    df = data[cat]
    lbl = CATEGORIES[cat]["label"]
    rows = []

    for band in BANDS:
        for is_dip, tag in [(False, "non-dipole"), (True, "dipole")]:
            sub = df[(df["r:band"] == band) & (df["is_dipole"] == is_dip)]
            if len(sub) == 0:
                continue
            psf = sub["psfFlux"].values
            diff = sub["diff_flux"].values
            with np.errstate(invalid="ignore", divide="ignore"):
                ratio = np.where(np.abs(diff) > 0, psf / diff, np.nan)
            rows.append(
                {
                    "category": lbl,
                    "band": band,
                    "type": tag,
                    "N": len(sub),
                    "med_psfFlux": np.nanmedian(psf),
                    "mad_psfFlux": mad(psf),
                    "med_diff": np.nanmedian(diff),
                    "mad_diff": mad(diff),
                    "med_ratio": np.nanmedian(ratio),
                }
            )

    df_stats = pd.DataFrame(rows)
    print(f"\n{'=' * 70}")
    print(f"  {lbl} — flux statistics (nJy)")
    print(f"{'=' * 70}")
    display(
        df_stats.style.format(
            {
                "med_psfFlux": "{:.1f}",
                "mad_psfFlux": "{:.1f}",
                "med_diff": "{:.1f}",
                "mad_diff": "{:.1f}",
                "med_ratio": "{:.3f}",
            }
        )
    )

## 9. Conclusions

Key questions answered by this notebook:

1. **Do dipole-flagged sources cluster away from the y = x line?**  
   Inspect the scatter plots in Section 6. Open circles far from the identity line
   confirm that dipole detection correlates with PSF-subtraction failures.

2. **Is the offset bright-magnitude-dependent (colour-coded)?**  
   Yellow points (bright stars, small AB mag) concentrated near the origin at low
   |psfFlux| but large |diff_flux| would indicate bright-template PSF-matching
   residuals unrelated to stellar variability.

3. **Do different stellar categories differ in the scatter plot morphology?**  
   Compare the three figures. The `Gaia variable` category should show a broader
   distribution on the y = x line (real flux changes), whereas stable-star categories
   should ideally cluster near the origin.

4. **What is the median ratio psfFlux / (scienceFlux − templateFlux)?**  
   A ratio systematically above or below 1 reveals a flux-scale offset between the
   two estimators (see the statistics table in Section 8).

> **Recommended next step:** for the most extreme outlier objects in the
> scatter plots (large |psfFlux| or large |diff_flux|) retrieve the Butler
> difference-image stamps (notebook 12b) and examine whether the dipole morphology
> is visible and which axis drives the discrepancy.

In [ ]:
print("Notebook 11b — dipole scatter plot — complete.")